In [1]:
!pip install qiskit qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 40.9 MB/s eta 0:00:00


## Code Demo 1: Manual Circuit Optimization (Canceling & Merging Gates)
This code demonstrates gate cancellation ($H \cdot H = I$) and rotation merging ($R_z(\theta_1) + R_z(\theta_2)$) programmatically in Qiskit using built-in optimization passes.

In [7]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, Optimize1qGatesDecomposition
from qiskit.circuit.library import HGate


# 1. Build an un-optimized abstract circuit
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)                  # H * H = Identity (Should cancel out)
qc.rz(np.pi / 4, 0)
qc.rz(np.pi / 4, 0)      # Rz(pi/4) + Rz(pi/4) = Rz(pi/2)

print("--- BEFORE OPTIMIZATION ---")
print(qc.draw(output='text'))

# 2. Run Optimization Pass Manager
# InverseCancellation handles H*H, Optimize1qGatesDecomposition merges Rz gates
pass_manager = PassManager([
    InverseCancellation([(HGate(), HGate())]),
    Optimize1qGatesDecomposition()
])

optimized_qc = pass_manager.run(qc)

print("\n--- AFTER OPTIMIZATION ---")
print(optimized_qc.draw(output='text'))

--- BEFORE OPTIMIZATION ---
   ┌───┐┌───┐┌─────────┐┌─────────┐
q: ┤ H ├┤ H ├┤ Rz(π/4) ├┤ Rz(π/4) ├
   └───┘└───┘└─────────┘└─────────┘

--- AFTER OPTIMIZATION ---
global phase: 7π/4
   ┌─────────┐
q: ┤ U1(π/2) ├
   └─────────┘


## Code Demo 2: Full Transpilation Pipeline (Targeting a QPU)
This code takes a non-native circuit and transpiles it for a specific fake/simulated IBM Quantum Processor (e.g., FakeSherbrooke), demonstrating unrolling to native gates, qubit mapping, and routing.

In [9]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# 1. Create a Fake/Simulated Hardware Backend (IBM 127-qubit Sherbrooke QPU)
backend = FakeSherbrooke()

# 2. Define an Abstract Circuit with high-level gates (Toffoli, SWAP, H)
qc = QuantumCircuit(3)
qc.h(0)
qc.swap(0, 2)  # Qubit 0 and Qubit 2 are not directly connected on physical chip!
qc.ccx(0, 1, 2) # Toffoli gate (3-qubit gate, definitely non-native)

print("--- ABSTRACT INPUT CIRCUIT ---")
print(qc.draw(output='text'))

# 3. Transpile the circuit targeting the backend with high optimization (Level 3)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
transpiled_qc = pm.run(qc)
print("--- TRANSPILED CIRCUIT ---")
print(transpiled_qc.draw(output='text'))

print("\n--- TRANSPILED HARDWARE-READY CIRCUIT ---")
# Print basic circuit stats showing basis translation
print(f"Original Gate Count: {dict(qc.count_ops())}")
print(f"Transpiled Gate Count: {dict(transpiled_qc.count_ops())}")
print(f"Target Hardware Native Basis: {backend.operation_names}")

--- ABSTRACT INPUT CIRCUIT ---
     ┌───┐        
q_0: ┤ H ├─X───■──
     └───┘ │   │  
q_1: ──────┼───■──
           │ ┌─┴─┐
q_2: ──────X─┤ X ├
             └───┘
--- TRANSPILED CIRCUIT ---
global phase: 11π/8
                                                             ┌──────┐»
q_1 -> 112 ──────────────────────────────────────────────────┤0     ├»
           ┌─────────┐┌────┐  ┌──────────┐                   │      │»
q_2 -> 125 ┤ Rz(π/4) ├┤ √X ├──┤ Rz(-π/2) ├───────────────────┤  Ecr ├»
           ├─────────┤├────┤┌─┴──────────┴┐┌────┐┌──────────┐│      │»
q_0 -> 126 ┤ Rz(π/2) ├┤ √X ├┤ Rz(-1.8925) ├┤ √X ├┤ Rz(-π/2) ├┤1     ├»
           └─────────┘└────┘└─────────────┘└────┘└──────────┘└──────┘»
«                ┌───┐                                                       »
«q_1 -> 112 ─────┤ X ├───────────────────────────────────────────────────────»
«                └───┘                                               ┌──────┐»
«q_2 -> 125 ───────────────────────────────────────────

## Code Demo 3: Comparing Transpiler Optimization Levels (0 vs 3)
You can show your students how Qiskit's optimization_level parameter (ranging from 0 to 3) impacts circuit depth and two-qubit gate count.

In [10]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

# Build a complex circuit
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.h(0)
qc.h(1)
qc.h(2)

print(f"Original Circuit Depth: {qc.depth()}")

# Level 0: Quick translation, NO optimization
pm_level0 = generate_preset_pass_manager(optimization_level=0, backend=backend)
qc_lvl0 = pm_level0.run(qc)

# Level 3: Heavy optimization, gate cancellation, best layout search
pm_level3 = generate_preset_pass_manager(optimization_level=3, backend=backend)
qc_lvl3 = pm_level3.run(qc)

print("\n--- RESULTS ---")
print(f"Level 0 Transpiled Depth: {qc_lvl0.depth()} | Total Gates: {sum(qc_lvl0.count_ops().values())}")
print(f"Level 3 Transpiled Depth: {qc_lvl3.depth()} | Total Gates: {sum(qc_lvl3.count_ops().values())}")

Original Circuit Depth: 4

--- RESULTS ---
Level 0 Transpiled Depth: 18 | Total Gates: 40
Level 3 Transpiled Depth: 9 | Total Gates: 20
